# 解答② DPO

> **講師用**: 演習 `ex_02_dpo.ipynb` の完全解答です。

In [ ]:
import os
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model
from trl import DPOTrainer, DPOConfig

os.environ.setdefault('HF_HOME', '/data/shared/hf_cache')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# 解答: preference データの例（参考）
my_preference_data = [
    {
        'prompt': 'NumPy でゼロ行列を作成する方法を教えてください。',
        'chosen': (
            'NumPy でゼロ行列を作成するには `np.zeros()` 関数を使います。\n\n'
            '```python\nimport numpy as np\n\n'
            '# 3x4 のゼロ行列\nmatrix = np.zeros((3, 4))\nprint(matrix.shape)  # (3, 4)\n```\n\n'
            'dtype を指定することもできます: `np.zeros((3, 4), dtype=int)`'
        ),
        'rejected': 'np.zeros を使います。',
    },
    {
        'prompt': 'Python の `__init__` メソッドは何をするものですか？',
        'chosen': (
            '`__init__` はクラスのコンストラクタメソッドで、インスタンス生成時に自動的に呼ばれます。\n\n'
            '```python\nclass Dog:\n    def __init__(self, name, age):\n'
            '        self.name = name\n        self.age = age\n\n'
            'dog = Dog("ポチ", 3)  # __init__ が呼ばれる\nprint(dog.name)  # ポチ\n```\n\n'
            'インスタンス変数の初期化に使うのが主な目的です。'
        ),
        'rejected': 'クラスを初期化するメソッドです。',
    },
    {
        'prompt': 'バッチ正規化（Batch Normalization）の役割は何ですか？',
        'chosen': (
            'バッチ正規化は各層への入力をミニバッチ単位で正規化する手法です。\n\n'
            '**効果**\n- 学習を安定化させる（勾配消失・爆発を軽減）\n'
            '- より大きな学習率を使えるようになる\n'
            '- 正則化効果があり過学習を抑制する\n\n'
            '各ミニバッチ内でゼロ平均・単位分散に正規化した後、学習可能なパラメータ γ, β でスケール・シフトします。'
        ),
        'rejected': '正規化することで学習が速くなります。',
    },
]

dataset = Dataset.from_list(my_preference_data)
print(f'データ件数: {len(dataset)}')

In [ ]:
# 解答: DPOConfig
BETA = 0.1
BASE_MODEL = 'meta-llama/Meta-Llama-3-8B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
) if device == 'cuda' else None

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto' if device == 'cuda' else None,
    torch_dtype=torch.bfloat16 if device == 'cuda' else torch.float32,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
)
model = get_peft_model(model, lora_config)

dpo_args = DPOConfig(
    output_dir='./outputs/ex02_dpo',
    max_steps=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    beta=BETA,
    bf16=device == 'cuda',
    logging_steps=5,
    report_to='none',
    max_length=512,
    max_prompt_length=256,
)

trainer = DPOTrainer(
    model=model,
    args=dpo_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)
trainer.train()
print('DPO 学習完了！')